In [1]:
from geopy.geocoders import Nominatim
import time
import openrouteservice
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import pandas as pd
import folium
from folium import plugins
from random import randint
import json
import re
from pprint import pprint
import numpy as np

In [ ]:
df = pd.read_csv('D:\Route optimization\csv files\locations_test2.csv')

# Get the postcodes as a list (preserve order)
Postcodes = df['Postcodes'].tolist()
Names = df['Names'].tolist()
drivers = df[df['Car'] == True].copy()
driver_postcodes = drivers['Postcodes'].tolist()

#Clean the Postcode column
df['Postcodes'] = df['Postcodes'].str.replace('\u202f', ' ', regex=False).str.strip()

# Normalize boolean columns if needed
df["Attendance"] = df["Attendance"]==True
df["Car"] = df["Car"].astype(bool)

# Filter only attendees
attendees = df[df["Attendance"] == True].copy()
attendee_postcodes = attendees["Postcodes"].tolist()

# Separate riders and possible drivers
riders = attendees[attendees["Car"] == True].copy()
# Filter for potential drivers: people who attend and have a car
potential_drivers = df[(df["Attendance"] == True) & (df["Car"] == True)].copy()

# Extract their car capacities
car_capacities = potential_drivers["Car capacity"].tolist()
car_capacities = [int(c) for c in car_capacities]

def clean_postcode(pc):
    if isinstance(pc, str):
        pc = re.sub(r'\s+', ' ', pc)
        return pc.strip()
    return pc
df['Postcodes'] = df['Postcodes'].apply(clean_postcode)
Postcodes = df['Postcodes']

print(car_capacities)


[4, 3, 4]


In [3]:
geolocator = Nominatim(user_agent="bristol_vrp")

coordinates = []
for p in attendee_postcodes:
    location = geolocator.geocode({
    'postalcode': p,
    'country': 'United Kingdom'
    })
    if location:
        coordinates.append((location.latitude, location.longitude))
    else:
        coordinates.append(None)
    time.sleep(1)  # avoid rate-limiting
node_locations = {
    i: {"lat": lat, "lon": lon}
    for i, (lat, lon) in enumerate(coordinates)
}

print(coordinates)


[(51.4627052, -2.6133195), (51.4800734, -2.512041), (51.4481924, -2.5422427), (51.5250604, -2.5957312), (51.4422992, -2.5629191), (51.4701948, -2.5948754), (51.4585453, -2.602144), (51.4548932, -2.5951641), (51.4539278, -2.5832661), (51.5471855, -2.4149734), (51.44813, -2.60178), (51.4584195, -2.5761907), (51.4444881, -2.615234), (51.4460932, -2.5511497), (51.4569398, -2.5523045), (51.4701632, -2.6005874), (51.47822, -2.58311), (51.4566472, -2.6150042), (51.4886086, -2.6133921), (51.5124395, -2.6192142), (51.4701632, -2.6005874), (51.40723, -2.61481), (51.41959, -2.57816), (51.4868967, -2.501593), (51.5273256, -2.4810112)]


In [4]:
print(node_locations)

{0: {'lat': 51.4627052, 'lon': -2.6133195}, 1: {'lat': 51.4800734, 'lon': -2.512041}, 2: {'lat': 51.4481924, 'lon': -2.5422427}, 3: {'lat': 51.5250604, 'lon': -2.5957312}, 4: {'lat': 51.4422992, 'lon': -2.5629191}, 5: {'lat': 51.4701948, 'lon': -2.5948754}, 6: {'lat': 51.4585453, 'lon': -2.602144}, 7: {'lat': 51.4548932, 'lon': -2.5951641}, 8: {'lat': 51.4539278, 'lon': -2.5832661}, 9: {'lat': 51.5471855, 'lon': -2.4149734}, 10: {'lat': 51.44813, 'lon': -2.60178}, 11: {'lat': 51.4584195, 'lon': -2.5761907}, 12: {'lat': 51.4444881, 'lon': -2.615234}, 13: {'lat': 51.4460932, 'lon': -2.5511497}, 14: {'lat': 51.4569398, 'lon': -2.5523045}, 15: {'lat': 51.4701632, 'lon': -2.6005874}, 16: {'lat': 51.47822, 'lon': -2.58311}, 17: {'lat': 51.4566472, 'lon': -2.6150042}, 18: {'lat': 51.4886086, 'lon': -2.6133921}, 19: {'lat': 51.5124395, 'lon': -2.6192142}, 20: {'lat': 51.4701632, 'lon': -2.6005874}, 21: {'lat': 51.40723, 'lon': -2.61481}, 22: {'lat': 51.41959, 'lon': -2.57816}, 23: {'lat': 51.4

In [5]:
client = openrouteservice.Client(key='eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6Ijc2OTExMGNhMzg1MjRhMGU5ZDE5OTA0ZDg0NjYzYTVmIiwiaCI6Im11cm11cjY0In0=')  # get one at openrouteservice.org

# ORS expects (lon, lat)
locations_lonlat = [(lon, lat) for (lat, lon) in coordinates]

matrix = client.distance_matrix(
    locations=locations_lonlat,
    profile='driving-car',
    metrics=['distance'],  # or 'distance'
    resolve_locations=True,  # road distance generation
    units='m'
)

distance_matrix = matrix['distances']

In [6]:
# Convert to a format suitable for OR-Tools
# distance matrix should be integer values
distance_matrix = [
    [round(value) for value in row]
    for row in distance_matrix
]

# Save the distance matrix to a JSON file, so it can be reused later, save time from re-fetching
#with open("distance_matrix.json", "w") as f: 
#    json.dump(distance_matrix, f)

print(np.array(distance_matrix))

[[    0  9442  7928  9691  6346  2086  1500  3320  3919 21721  2509  3762
   4521  7725  6699  1655  4324   860  3609  7123  1655  9973  9693 12980
  15471]
 [ 9294     0  7444 16233  7520  7453  8105  7521  6996 12429  8590  6516
  10523  7241  5223  7900  6315  9293 11310 16861  7900 17514 14155  1346
   8065]
 [ 7883  7436     0 23364  2976  7261  6694  6109  4244 22910  7179  5534
   7171   908  2218  7707  7357  7882 10105 23993  7707  8423  5064  8160
  18546]
 [ 9492 16533 22045     0 21936  9373 10606 20715 20190 16123 11616 19710
  14887 21842 19823  9450  8967 10340  6790  3387  9450 26924 25252 15871
  11217]
 [ 5990  7515  2980 21345     0  5241  4801  4216  3301 21235  5286  3514
   4195  1299  2297  5688  5337  5989  8085 21973  5688  7562  4923  8239
  14985]
 [ 2086  7445  7200  9528  5383     0  1858  2223  2651 19723  3403  2495
   6172  6996  4670   463  2164  2775  2938  6960   463 11624  7714 10983
  13473]
 [ 1510  8116  6601 20504  5019  1609     0  1994  2592 20

In [7]:
num_vehicles = len(potential_drivers)

def create_data_model():
    data = {}
    data['distance_matrix'] = distance_matrix  # from ORS
    data['num_vehicles'] = num_vehicles  # or however many
    data['starts'] = [0]* data['num_vehicles']  # assuming BS8 2ES is first
    data['ends'] = drivers.index[:num_vehicles].tolist()
    data['demands'] = [1]* (len(Postcodes))  # depot has demand 0, each stop has demand 1  
    data['vehicle_capacities'] = car_capacities#[4] * data['num_vehicles']  # max 4 stops per vehicle
    data['depot'] = 0  # assuming depot is the first location

    # Check you have enough end locations
    #if len(drivers) < num_vehicles:
    #    raise ValueError("Not enough 'car' == 'yes' locations to assign one per vehicle")

    # Demand: 0 for depot, 1 for each stop (count visits)
    #data['vehicle_capacities'] = [4] * data['num_vehicles'] # max 4 stops per vehicle
    return data

In [8]:
data=create_data_model()

In [9]:
print(data['distance_matrix'])
print(data['starts'])
print(data['ends'])
print(data['demands'])
print(data['vehicle_capacities'])
print(len(data['distance_matrix']), len(data['demands']))

[[0, 9442, 7928, 9691, 6346, 2086, 1500, 3320, 3919, 21721, 2509, 3762, 4521, 7725, 6699, 1655, 4324, 860, 3609, 7123, 1655, 9973, 9693, 12980, 15471], [9294, 0, 7444, 16233, 7520, 7453, 8105, 7521, 6996, 12429, 8590, 6516, 10523, 7241, 5223, 7900, 6315, 9293, 11310, 16861, 7900, 17514, 14155, 1346, 8065], [7883, 7436, 0, 23364, 2976, 7261, 6694, 6109, 4244, 22910, 7179, 5534, 7171, 908, 2218, 7707, 7357, 7882, 10105, 23993, 7707, 8423, 5064, 8160, 18546], [9492, 16533, 22045, 0, 21936, 9373, 10606, 20715, 20190, 16123, 11616, 19710, 14887, 21842, 19823, 9450, 8967, 10340, 6790, 3387, 9450, 26924, 25252, 15871, 11217], [5990, 7515, 2980, 21345, 0, 5241, 4801, 4216, 3301, 21235, 5286, 3514, 4195, 1299, 2297, 5688, 5337, 5989, 8085, 21973, 5688, 7562, 4923, 8239, 14985], [2086, 7445, 7200, 9528, 5383, 0, 1858, 2223, 2651, 19723, 3403, 2495, 6172, 6996, 4670, 463, 2164, 2775, 2938, 6960, 463, 11624, 7714, 10983, 13473], [1510, 8116, 6601, 20504, 5019, 1609, 0, 1994, 2592, 20394, 1861, 243

In [10]:
# -----------------------------------------------
# HARD CONSTRAINT: remove stops too close to depot
# -----------------------------------------------
depot = data["starts"][0]

# Calculate distances from depot
distances_from_depot = [
    data["distance_matrix"][depot][node]
    for node in range(len(data["distance_matrix"]))
    if node not in (depot,)  # skip depot itself
]

# Define dynamic threshold (e.g., 70% of average)
#average_distance = sum(distances_from_depot) / len(distances_from_depot)
MIN_DISTANCE_FROM_DEPOT = 3000  #0.7 * average_distance

# Filter nodes: keep depot, ends, and valid stops
valid_nodes = [
    node for node in range(len(data["distance_matrix"]))
    if node == depot or node in data["ends"] or data["distance_matrix"][depot][node] > MIN_DISTANCE_FROM_DEPOT
]

# Rebuild distance matrix and demands using only valid nodes
data["distance_matrix"] = [[data["distance_matrix"][i][j] for j in valid_nodes] for i in valid_nodes]
data["demands"] = [data["demands"][i] for i in valid_nodes]

# Update starts/ends mapping to match filtered indices
data["starts"] = [valid_nodes.index(s) for s in data["starts"]]
data["ends"] = [valid_nodes.index(e) for e in data["ends"]]


In [11]:
print(valid_nodes)
print(len(valid_nodes))
print(len(data["distance_matrix"]))

[0, 1, 2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 16, 18, 19, 21, 22, 23, 24]
19
19


In [ ]:
def print_solution(data, manager, routing, assignment):
    """Prints assignment on console."""
    print(f"Objective: {assignment.ObjectiveValue()}")
    # Display dropped nodes.
    dropped_nodes = "Dropped nodes:"
    for node in range(routing.Size()):
        if routing.IsStart(node) or routing.IsEnd(node):
            continue
        if assignment.Value(routing.NextVar(node)) == node:
            dropped_nodes += f" {manager.IndexToNode(node)}"
    print(dropped_nodes)
    # Display routes
    total_distance = 0
    total_load = 0
    for vehicle_id in range(data["num_vehicles"]):
        #routing.SetFixedCostOfVehicle(0, vehicle_id) # Set fixed cost to 0 for all vehicles to force usage

        if not routing.IsVehicleUsed(assignment, vehicle_id):
            continue
        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data["demands"][node_index]
            plan_output += f" {node_index} Load({route_load}) -> "
            previous_index = index
            index = assignment.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
        plan_output += f" {manager.IndexToNode(index)} Load({route_load})\n"
        plan_output += f"Distance of the route: {route_distance}m\n"
        plan_output += f"Load of the route: {route_load}\n"
        print(plan_output)
        total_distance += route_distance
        total_load += route_load
    print(f"Total Distance of all routes: {total_distance}m")
    print(f"Total Load of all routes: {total_load}")

In [13]:
import folium

def plot_routes(data, manager, routing, assignment, node_locations):
    # Start the map at the depot
    depot_coords = (node_locations[0]["lat"], node_locations[0]["lon"])
    m = folium.Map(location=depot_coords, zoom_start=12)

    colors = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue"]

    for vehicle_id in range(data["num_vehicles"]):
        if not routing.IsVehicleUsed(assignment, vehicle_id):
            continue
        index = routing.Start(vehicle_id)
        route = []
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            coord = (node_locations[node_index]["lat"], node_locations[node_index]["lon"])
            route.append(coord)
            index = assignment.Value(routing.NextVar(index))
        node_index = manager.IndexToNode(index)
        route.append((node_locations[node_index]["lat"], node_locations[node_index]["lon"]))

        # Draw polyline
        folium.PolyLine(route, color=colors[vehicle_id % len(colors)],
                        weight=5, opacity=0.7, tooltip=f"Vehicle {vehicle_id}").add_to(m)

        # Mark stops
        for i, coord in enumerate(route):
            folium.CircleMarker(
                location=coord,
                radius=5,
                color=colors[vehicle_id % len(colors)],
                fill=True,
                fill_opacity=0.9,
                popup=f"Vehicle {vehicle_id} Stop {i}"
            ).add_to(m)

    # Show dropped nodes if any
    for node in range(1, routing.Size()):
        if assignment.Value(routing.NextVar(node)) == node:
            dropped_node = manager.IndexToNode(node)
            coord = (node_locations[dropped_node]["lat"], node_locations[dropped_node]["lon"])
            folium.Marker(
                location=coord,
                icon=folium.Icon(color="gray"),
                popup=f"Dropped: {dropped_node}",
            ).add_to(m)

    m.save("routes_map.html")



In [14]:
# Precompute distances to final stops for each node (for bias)
distance_to_final = []
for node_idx in range(len(data["distance_matrix"])):
    # Assuming a single end point per vehicle or same end for all vehicles
    # If multiple ends, you might need to calculate per-vehicle later
    end_node = data["ends"][0]  
    distance_to_final.append(data["distance_matrix"][node_idx][end_node])


In [15]:
def main():
    """Solve the CVRP problem."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["starts"], data["ends"])
    # was data["depot"]
    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)

        # Create and register a transit callback.
    ALPHA = 0.8
    BETA = 0.3

    # Create a separate cost callback for each vehicle
    for vehicle_id in range(data["num_vehicles"]):
        end_node = data["ends"][vehicle_id]

        def make_cost_callback(end_node=end_node):
            def cost_callback(from_index, to_index):
                from_node = manager.IndexToNode(from_index)
                to_node = manager.IndexToNode(to_index)

                base_distance = data["distance_matrix"][from_node][to_node]
                closeness_cost = data["distance_matrix"][to_node][end_node]

                return int(ALPHA * closeness_cost + BETA * base_distance)
            return cost_callback

        transit_callback_index = routing.RegisterTransitCallback(make_cost_callback())
        routing.SetArcCostEvaluatorOfVehicle(transit_callback_index, vehicle_id)

    # Add Capacity constraint.
    def demand_callback(from_index):
        """Returns the demand of the node."""
        # Convert from routing variable Index to demands NodeIndex.
        from_node = manager.IndexToNode(from_index)
        return data["demands"][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # null capacity slack
        data["vehicle_capacities"],  # vehicle maximum capacities
        True,  # start cumul to zero
        "Capacity",
    )

    # Add "VisitCount" dimension to cap visits
    def ones_callback(from_index):
        return 1  # Each node counts as 1 visit

    ones_index = routing.RegisterUnaryTransitCallback(ones_callback)
    routing.AddDimensionWithVehicleCapacity(
        ones_index,
        0,  # no slack
        [5] * data['num_vehicles'],  # max 5 route points incl. depot
        True,  # start cumul to zero
        "VisitCount"
    )

    # Optionally: Force max 5 nodes including depot → only 4 stops
    visit_dim = routing.GetDimensionOrDie("VisitCount")
    for vehicle_id in range(data['num_vehicles']):
        visit_dim.CumulVar(routing.End(vehicle_id)).SetMax(5)


    # Allow to drop nodes.
    penalty = 1000000 # Penalty for dropping a node, adjust the distance as needed

    #average_distance = sum(sum(r) for r in data["distance_matrix"]) / (len(data["distance_matrix"])**2)
    #penalty = int(2 * average_distance)  # about 2x average travel cost
    #for node in range(1, len(data["distance_matrix"])):
    #    routing.AddDisjunction([manager.NodeToIndex(node)], penalty)
    fixed_nodes = set(data["starts"] + data["ends"])
    #This ensures only the intermediate stops are droppable, not the required start/end points.
    for node in range(1, len(data["distance_matrix"])):
        if node not in fixed_nodes:
            routing.AddDisjunction([manager.NodeToIndex(node)], penalty)

    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.FromSeconds(5)

    # Solve the problem.
    assignment = routing.SolveWithParameters(search_parameters)

    # Print solution on console.
    if assignment:
        print_solution(data, manager, routing, assignment)
        plot_routes(data, manager, routing, assignment, node_locations)
if __name__ == "__main__":
    main()

Objective: 13040736
Dropped nodes: 5 6 7 8 9 10 12 16 17 20 21 22 24
Route for vehicle 0:
 0 Load(1) ->  11 Load(2) ->  14 Load(3) ->  23 Load(4) ->  1 Load(4)
Distance of the route: 14605m
Load of the route: 4

Route for vehicle 1:
 0 Load(1) ->  4 Load(2) ->  13 Load(3) ->  2 Load(3)
Distance of the route: 5675m
Load of the route: 3

Route for vehicle 2:
 0 Load(1) ->  15 Load(2) ->  18 Load(3) ->  19 Load(4) ->  3 Load(4)
Distance of the route: 20456m
Load of the route: 4

Total Distance of all routes: 40736m
Total Load of all routes: 11
